# Gastrula-to-Pup data download and preparation

This notebook downloads and prepares the **Qiu et al. 2024** mouse prenatal development dataset
([Nature 2024](https://www.nature.com/articles/s41586-024-07069-w)) for multi-modal analysis
with `RegularizedMultimodalVI`.

**Dataset summary**:
- **Protocol**: Optimised sci-RNA-seq3 (single-nucleus, combinatorial indexing)
- **Scale**: ~12.4M nuclei, ~45K genes, 74 embryos, 15 experimental batches
- **Developmental stages**: E8.0–P0 (gastrula to birth)
- **3 modalities**: total RNA (h5ad), exon/spliced (RDS), intron/unspliced (RDS)

**Data sources**:
- [Authors' download portal](https://omg.gs.washington.edu/jax/public/download.html)
- [CellxGene](https://cellxgene.cziscience.com/collections/45d5d2c3-bc28-4814-aed6-0bb6f0e11c82)
- [GEO GSE228590](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE228590)

**Goal**: Download all data, merge into a single AnnData with 3 layers (total, spliced, unspliced),
extract metadata (including `pcr_well` from barcodes), compute QC metrics, and save an
unfiltered checkpoint for downstream analysis.

In [ ]:
import gc
import glob
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Download data from authors' portal

The data is hosted at `shendure-web.gs.washington.edu` and consists of:

| Data | Files | Size | Format |
|------|-------|------|--------|
| Total RNA counts | `adata_JAX_dataset_{1,2,3,4}.h5ad` | ~42 GB | AnnData (h5ad) |
| Cell metadata | `df_cell.csv` | 1.5 GB | CSV |
| Gene metadata | `df_gene.csv` | 1.7 MB | CSV |
| Exon/intron counts | `embryo_{1..74}_exp_{exon,intron}.rds` | ~47 GB | R sparse matrices (RDS) |

The download uses `wget -c` for resume support (important for large files over slow connections).

In [ ]:
import subprocess

BASE_URL = "https://shendure-web.gs.washington.edu/content/members/cxqiu/public/backup/jax/download"

# --- h5ad files (total RNA) + metadata CSVs ---
adata_files = [
    "adata/adata_JAX_dataset_1.h5ad",
    "adata/adata_JAX_dataset_2.h5ad",
    "adata/adata_JAX_dataset_3.h5ad",
    "adata/adata_JAX_dataset_4.h5ad",
    "adata/df_cell.csv",
    "adata/df_gene.csv",
]

for f in adata_files:
    dest = os.path.join(DATA_DIR, os.path.basename(f))
    if not os.path.exists(dest):
        url = f"{BASE_URL}/{f}"
        print(f"Downloading {os.path.basename(f)}...")
        subprocess.run(["wget", "-c", url, "-O", dest], check=True)
    else:
        print(f"Already exists: {os.path.basename(f)}")

In [ ]:
# --- Exon/intron RDS files (148 files: 74 embryos x 2 modalities) ---
rds_dir = os.path.join(DATA_DIR, "embryo_exon_intron")
os.makedirs(rds_dir, exist_ok=True)

for i in range(1, 75):
    for modality in ["exon", "intron"]:
        fname = f"embryo_{i}_exp_{modality}.rds"
        dest = os.path.join(rds_dir, fname)
        if not os.path.exists(dest):
            url = f"{BASE_URL}/embryo_exon_intron/{fname}"
            print(f"Downloading {fname}...")
            subprocess.run(["wget", "-c", url, "-O", dest], check=True)

rds_files = sorted(glob.glob(os.path.join(rds_dir, "*.rds")))
print(f"\nTotal RDS files: {len(rds_files)}")

## 2. Load total RNA h5ad files

The 4 h5ad files contain the total RNA count matrix split across chunks.
We load and concatenate them into a single AnnData.

In [ ]:
h5ad_paths = sorted(glob.glob(os.path.join(DATA_DIR, "adata_JAX_dataset_*.h5ad")))
print(f"Found {len(h5ad_paths)} h5ad files")

adatas = []
for path in h5ad_paths:
    print(f"Loading {os.path.basename(path)}...")
    adata_chunk = sc.read_h5ad(path)
    adatas.append(adata_chunk)
    print(f"  {adata_chunk.shape}")

adata = ad.concat(adatas, join="outer")
del adatas
gc.collect()
print(f"\nTotal RNA AnnData: {adata.shape}")

## 3. Merge cell metadata

Load `df_cell.csv` (cell annotations: embryo_id, day, experimental_batch, celltype, etc.)
and `df_gene.csv` (gene annotations) and merge into the AnnData.

In [ ]:
df_cell = pd.read_csv(os.path.join(DATA_DIR, "df_cell.csv"))
df_gene = pd.read_csv(os.path.join(DATA_DIR, "df_gene.csv"))

print(f"df_cell: {df_cell.shape}")
print(f"  columns: {list(df_cell.columns)}")
print(f"\ndf_gene: {df_gene.shape}")
print(f"  columns: {list(df_gene.columns)}")

# Map cell annotations to adata.obs
df_cell = df_cell.set_index("cell_id")
adata.obs = adata.obs.join(df_cell, how="left")

print(f"\nAnnData obs columns after merge: {list(adata.obs.columns)}")
print(f"Cells with metadata: {adata.obs['embryo_id'].notna().sum()} / {adata.n_obs}")

## 4. Extract `pcr_well` from cell barcodes

sci-RNA-seq3 uses **3-level combinatorial indexing**. The cell barcode encodes the PCR well
(Level 3), which determines sequencing depth and protease-step ambient RNA.

Barcode format: `run_4_P2-01A.ATTCAAGCATGTTACGCAAG-0-0`
- `run_4` = sequencing run
- `P2-01A` = PCR well (plate-row-column) → this is what we extract
- `ATTCAAGCATGTTACGCAAG` = combinatorial barcode (RT + ligation)
- `-0-0` = suffix

Multiple embryos (~12) share each PCR well. The PCR well determines:
- Sequencing budget allocation (`library_size_key`)
- Protease-step ambient contamination (`ambient_covariate_keys`)

In [ ]:
# Extract PCR well from cell barcode
# Format: run_X_PLATE-WELL.BARCODE-suffix
# Example: run_4_P2-01A.ATTCAAGCATGTTACGCAAG-0-0 -> pcr_well = "P2-01A"
pattern = r"run_\d+_([^.]+)\."
adata.obs["pcr_well"] = adata.obs_names.str.extract(pattern, expand=False)

# Also extract run_id for reference
adata.obs["run_id"] = adata.obs_names.str.extract(r"(run_\d+)_", expand=False)

print(f"Unique pcr_wells: {adata.obs['pcr_well'].nunique()}")
print(f"Unique runs: {adata.obs['run_id'].nunique()}")
print(f"Unique embryos: {adata.obs['embryo_id'].nunique()}")
print(f"Unique experiments: {adata.obs['experimental_batch'].nunique()}")

# Cross-tabulation: how many embryos per PCR well?
print("\nEmbryos per PCR well:")
print(adata.obs.groupby("pcr_well")["embryo_id"].nunique().describe())

In [ ]:
# Create composite dispersion column
# dispersion_key only accepts a single str, so we create a composite of embryo_id x experimental_batch
adata.obs["embryo_experiment"] = adata.obs["embryo_id"].astype(str) + "_" + adata.obs["experimental_batch"].astype(str)
print(f"Unique embryo_experiment groups: {adata.obs['embryo_experiment'].nunique()}")

## 5. Gene identifiers

Check the gene identifier format and map using `df_gene.csv` if needed.

In [ ]:
# Inspect gene identifiers
print(f"First 5 var_names: {list(adata.var_names[:5])}")
print(f"var columns: {list(adata.var.columns)}")
print("\ndf_gene first rows:")
print(df_gene.head())

# If h5ad uses Ensembl IDs and df_gene has symbol mapping, merge it
# Exact implementation depends on the actual df_gene.csv structure
# This will be adapted at runtime based on what we find
if "gene_short_name" in df_gene.columns and "gene_id" in df_gene.columns:
    gene_map = df_gene.set_index("gene_id")["gene_short_name"]
    adata.var["gene_symbol"] = adata.var_names.map(gene_map)
    print(f"\nMapped gene symbols: {adata.var['gene_symbol'].notna().sum()} / {adata.n_vars}")

## 6. Load exon/intron RDS files

The 148 per-embryo RDS files contain **exon** (spliced) and **intron** (unspliced) count
matrices separately. Each file is a sparse matrix (dgCMatrix) with cells as rows and genes
as columns.

We use `rds2py` (pure Python, no R dependency) to load these files.

In [ ]:
from rds2py import read_rds

# Load first file to understand structure
test_exon = read_rds(os.path.join(rds_dir, "embryo_1_exp_exon.rds"))
print(f"Type: {type(test_exon)}")
print(f"Shape: {test_exon.shape if hasattr(test_exon, 'shape') else 'N/A'}")

# If it's a scipy sparse matrix, inspect format
if sp.issparse(test_exon):
    print(f"Sparse format: {test_exon.format}")
    print(f"NNZ: {test_exon.nnz}")
    print(f"Dtype: {test_exon.dtype}")
elif hasattr(test_exon, "__dict__"):
    print(f"Attributes: {list(test_exon.__dict__.keys()) if hasattr(test_exon, '__dict__') else dir(test_exon)}")

# Also check if it's an anndata.AnnData or has row/col names
if isinstance(test_exon, ad.AnnData):
    print(f"AnnData shape: {test_exon.shape}")
    print(f"obs_names[:5]: {list(test_exon.obs_names[:5])}")
    print(f"var_names[:5]: {list(test_exon.var_names[:5])}")

del test_exon
gc.collect()

In [ ]:
# Load all exon (spliced) and intron (unspliced) matrices
# NOTE: The exact extraction code depends on the rds2py object structure
# found in the cell above. The code below handles the most common cases:
# 1. Direct sparse matrix (dgCMatrix)
# 2. AnnData object
# 3. Named list with matrix + rownames/colnames

spliced_adatas = []
unspliced_adatas = []

for i in range(1, 75):
    if i % 10 == 0:
        print(f"Loading embryo {i}/74...")

    exon_path = os.path.join(rds_dir, f"embryo_{i}_exp_exon.rds")
    intron_path = os.path.join(rds_dir, f"embryo_{i}_exp_intron.rds")

    exon_obj = read_rds(exon_path)
    intron_obj = read_rds(intron_path)

    # Convert to AnnData if needed
    # Case 1: Already AnnData
    if isinstance(exon_obj, ad.AnnData):
        spliced_adatas.append(exon_obj)
        unspliced_adatas.append(intron_obj)
    # Case 2: Sparse matrix with separate row/col names
    elif sp.issparse(exon_obj):
        # rds2py may store rownames/colnames as attributes
        spliced_adatas.append(ad.AnnData(X=exon_obj))
        unspliced_adatas.append(ad.AnnData(X=intron_obj))
    # Case 3: dict-like with matrix and names
    elif hasattr(exon_obj, "keys") or isinstance(exon_obj, dict):
        # Try to extract matrix and names
        mat = exon_obj.get("data", exon_obj.get("x", exon_obj))
        if sp.issparse(mat) or isinstance(mat, np.ndarray):
            spliced_adatas.append(ad.AnnData(X=mat))
        mat_i = intron_obj.get("data", intron_obj.get("x", intron_obj))
        if sp.issparse(mat_i) or isinstance(mat_i, np.ndarray):
            unspliced_adatas.append(ad.AnnData(X=mat_i))
    else:
        print(f"WARNING: Unexpected type for embryo_{i}: {type(exon_obj)}")
        print(f"  dir: {dir(exon_obj)[:20]}")
        break

    del exon_obj, intron_obj

gc.collect()
print(f"\nLoaded {len(spliced_adatas)} exon + {len(unspliced_adatas)} intron matrices")

In [ ]:
# Concatenate into single AnnData objects
adata_spliced = ad.concat(spliced_adatas, join="outer")
del spliced_adatas
gc.collect()
print(f"Spliced AnnData: {adata_spliced.shape}")

adata_unspliced = ad.concat(unspliced_adatas, join="outer")
del unspliced_adatas
gc.collect()
print(f"Unspliced AnnData: {adata_unspliced.shape}")

## 7. Align cells and genes across modalities

The total RNA h5ad and exon/intron RDS files may have different cell/gene orderings.
We align them to the intersection of shared cells and genes.

In [ ]:
# Find shared cells
shared_cells = adata.obs_names.intersection(adata_spliced.obs_names)
print(f"Shared cells: {len(shared_cells)} / {adata.n_obs} (total) / {adata_spliced.n_obs} (spliced)")

# Find shared genes
shared_genes = adata.var_names.intersection(adata_spliced.var_names)
print(f"Shared genes: {len(shared_genes)} / {adata.n_vars} (total) / {adata_spliced.n_vars} (spliced)")

# Subset to shared cells and genes
adata = adata[shared_cells, shared_genes].copy()
adata_spliced = adata_spliced[shared_cells, shared_genes].copy()
adata_unspliced = adata_unspliced[shared_cells, shared_genes].copy()

print("\nAligned shapes:")
print(f"  Total RNA: {adata.shape}")
print(f"  Spliced: {adata_spliced.shape}")
print(f"  Unspliced: {adata_unspliced.shape}")

## 8. Compute QC metrics

Compute standard QC metrics on the total RNA counts:
- Total UMI counts per cell
- Number of genes detected per cell
- Mitochondrial fraction (mouse: `mt-` or `Mt-` prefix)
- Doublet scores (Scrublet, per experimental_batch)

In [ ]:
# Ensure counts are in .X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

# Basic QC
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Mitochondrial fraction (mouse gene names use mt- or Mt- prefix)
mt_genes = adata.var_names.str.startswith("mt-") | adata.var_names.str.startswith("Mt-")
mt_counts = np.array(adata[:, mt_genes].X.sum(1)).squeeze()
adata.obs["mt_frac"] = mt_counts / adata.obs["total_counts"]

print(f"MT genes found: {mt_genes.sum()}")
print(f"MT fraction: {adata.obs['mt_frac'].describe()}")
print(f"\nTotal counts: {adata.obs['total_counts'].describe()}")
print(f"\nN genes: {adata.obs['n_genes_by_counts'].describe()}")

In [ ]:
from regularizedvi.utils import filter_genes

# Select genes for Scrublet (on total RNA)
selected_for_scrub = filter_genes(
    adata,
    cell_count_cutoff=15,
    cell_percentage_cutoff2=0.05,
    nonz_mean_cutoff=1.07,
)
adata_for_scrublet = adata[:, selected_for_scrub].copy()

# Run Scrublet per experimental_batch (15 levels, manageable)
# Using experimental_batch rather than embryo_id (74 levels) to avoid too-small batches
sc.pp.scrublet(
    adata_for_scrublet,
    batch_key="experimental_batch",
    n_prin_comps=40,
    verbose=True,
)
adata.obs["doublet_score"] = adata_for_scrublet.obs["doublet_score"]
del adata_for_scrublet
gc.collect()

print("\nDoublet score distribution:")
print(adata.obs["doublet_score"].describe())

## 9. QC visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

qc_cols = ["total_counts", "n_genes_by_counts", "mt_frac", "doublet_score"]
for ax, col in zip(axes.flat[:4], qc_cols, strict=False):
    if col in adata.obs.columns:
        adata.obs[col].hist(ax=ax, bins=100)
        ax.set_title(col)
        ax.set_ylabel("Cells")

# Cells per embryo
ax = axes[1, 0]
cells_per_embryo = adata.obs.groupby("embryo_id").size().sort_values()
ax.barh(range(len(cells_per_embryo)), cells_per_embryo.values)
ax.set_xlabel("Cells")
ax.set_title(f"Cells per embryo (n={len(cells_per_embryo)})")
ax.set_yticks([])

# Cells per experimental_batch
ax = axes[1, 1]
cells_per_batch = adata.obs.groupby("experimental_batch").size().sort_values()
ax.barh(range(len(cells_per_batch)), cells_per_batch.values)
ax.set_yticks(range(len(cells_per_batch)))
ax.set_yticklabels(cells_per_batch.index, fontsize=8)
ax.set_xlabel("Cells")
ax.set_title(f"Cells per experimental_batch (n={len(cells_per_batch)})")

# Hide unused subplot
axes[1, 2].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Metadata summary tables
print("Cells per embryo:")
print(adata.obs.groupby("embryo_id").size().describe())

print("\nCells per PCR well:")
print(adata.obs.groupby("pcr_well").size().describe())

print("\nCells per experimental_batch:")
print(adata.obs.groupby("experimental_batch").size().describe())

print("\nEmbryos per developmental day:")
print(adata.obs.groupby("day")["embryo_id"].nunique())

## 10. Save unfiltered checkpoint

Save the full unfiltered AnnData with all metadata and QC metrics.
Spliced and unspliced counts are stored as layers alongside the total RNA counts.
This avoids re-running upstream steps when experimenting with filtering thresholds.

In [ ]:
# Save spliced/unspliced as layers in main adata
adata.layers["total"] = adata.X.copy()
adata.layers["spliced"] = adata_spliced.X.copy()
adata.layers["unspliced"] = adata_unspliced.X.copy()

# Verify
print(f"Layers: {list(adata.layers.keys())}")
print(f"Shape: {adata.shape}")
print(f"Obs columns: {list(adata.obs.columns)}")

# Save checkpoint
checkpoint_path = os.path.join(DATA_DIR, "gastrula_to_pup_unfiltered.h5ad")
adata.write_h5ad(checkpoint_path)
print(f"\nSaved unfiltered checkpoint: {checkpoint_path}")
print(f"File size: {os.path.getsize(checkpoint_path) / 1e9:.1f} GB")